<a href="https://colab.research.google.com/github/RaMarWilson1/GenAI/blob/main/HW5/Problem1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
VOCAB_SIZE = 10000
MAX_LEN = 200
EMBEDDING_DIM = 100
N_UNITS = 128
VALIDATION_SPLIT = 0.2
SEED = 42
LOAD_MODEL = False
BATCH_SIZE = 32
EPOCHS = 25

In [14]:
import requests
import re
import string
!pip install tensorflow
import tensorflow as tf
# List of URLs for additional texts (e.g., different Shakespeare plays)
urls = [
    "https://www.gutenberg.org/files/1041/1041-0.txt",  # Hamlet
    "https://www.gutenberg.org/files/152/152-0.txt",   # Macbeth
    "https://www.gutenberg.org/files/1112/1112-0.txt"   # Othello
]

# Initialize an empty string to hold all text
all_text = ""

# Download each text file and append to all_text
for url in urls:
    response = requests.get(url)
    text = response.text
    all_text += text + "\n\n"  # Separate texts by newlines

# Save combined text to a single file
with open("combined_shakespeare.txt", "w", encoding="utf-8") as file:
    file.write(all_text)

# Pad the punctuation, to treat them as separate 'words'
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}])", r" \1 ", s)
    s = re.sub(" +", " ", s)
    return s


# **Change 1:** Load data from 'combined_shakespeare.txt' and split into lines
with open("combined_shakespeare.txt", "r", encoding="utf-8") as file:
    filtered_data = file.readlines()

# **Change 2:** Apply `pad_punctuation` to each line
text_data = [pad_punctuation(x) for x in filtered_data]

# Display an example
example_data = text_data[9]
print(example_data)

In [12]:
# Convert to a Tensorflow Dataset
text_ds = (
    tf.data.Dataset.from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(1000)
)

In [17]:
# Create a vectorisation layer
vectorize_layer = tf.keras.layers.TextVectorization(
    standardize="lower",
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN + 1,
)

In [18]:
# Adapt the layer to the training set
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()

In [19]:
for i, word in enumerate(vocab[:10]):
    print(f"{i}: {word}")

0: 
1: [UNK]
2: ,
3: .
4: and
5: the
6: to
7: i
8: of
9: my


In [20]:
# Display the same example converted to ints
example_tokenised = vectorize_layer(example_data)
print(example_tokenised.numpy())

[]


In [21]:
def prepare_inputs(text):
    text = tf.expand_dims(text, -1)
    tokenized_sentences = vectorize_layer(text)
    x = tokenized_sentences[:, :-1]
    y = tokenized_sentences[:, 1:]
    return x, y


train_ds = text_ds.map(prepare_inputs)

In [23]:
inputs = tf.keras.layers.Input(shape=(None,), dtype="int32")
x = tf.keras.layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x = tf.keras.layers.LSTM(N_UNITS, return_sequences=True)(x)
outputs = tf.keras.layers.Dense(VOCAB_SIZE, activation="softmax")(x)

# Assuming 'models' refers to tf.keras.models
from tensorflow import keras
lstm = keras.models.Model(inputs, outputs)
lstm.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, None)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ embedding (Embedding)                │ (None, None, 100)           │       1,000,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, None, 128)           │         117,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, None, 10000)         │       1,290,000 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,407,248 (9.18 MB)

 Trainable params: 2,407,248 (9.18 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
lstm.compile("adam", loss_fn)

In [26]:
# Create a TextGenerator checkpoint
class TextGenerator(callbacks.Callback):
    def __init__(self, index_to_word, top_k=10):
        self.index_to_word = index_to_word
        self.word_to_index = {
            word: index for index, word in enumerate(index_to_word)
        }  # <1>

    def sample_from(self, probs, temperature):  # <2>
        probs = probs ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature):
        start_tokens = [
            self.word_to_index.get(x, 1) for x in start_prompt.split()
        ]  # <3>
        sample_token = None
        info = []
        while len(start_tokens) < max_tokens and sample_token != 0:  # <4>
            x = np.array([start_tokens])
            y = self.model.predict(x, verbose=0)  # <5>
            sample_token, probs = self.sample_from(y[0][-1], temperature)  # <6>
            info.append({"prompt": start_prompt, "word_probs": probs})
            start_tokens.append(sample_token)  # <7>
            start_prompt = start_prompt + " " + self.index_to_word[sample_token]
        print(f"\ngenerated text:\n{start_prompt}\n")
        return info

    def on_epoch_end(self, epoch, logs=None):
        self.generate("recipe for", max_tokens=100, temperature=1.0)

NameError: name 'callbacks' is not defined

In [28]:
# Create a TextGenerator checkpoint
from tensorflow.keras import callbacks # Import callbacks from tensorflow.keras

class TextGenerator(callbacks.Callback):
    def __init__(self, index_to_word, top_k=10):
        self.index_to_word = index_to_word
        self.word_to_index = {
            word: index for index, word in enumerate(index_to_word)
        }  # <1>

    def sample_from(self, probs, temperature):  # <2>
        probs = probs ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature):
        start_tokens = [
            self.word_to_index.get(x, 1) for x in start_prompt.split()
        ]  # <3>
        sample_token = None
        info = []
        while len(start_tokens) < max_tokens and sample_token != 0:  # <4>
            x = np.array([start_tokens])
            y = self.model.predict(x, verbose=0)  # <5>
            sample_token, probs = self.sample_from(y[0][-1], temperature)  # <6>
            info.append({"prompt": start_prompt, "word_probs": probs})
            start_tokens.append(sample_token)  # <7>
            start_prompt = start_prompt + " " + self.index_to_word[sample_token]
        print(f"\ngenerated text:\n{start_prompt}\n")
        return info

    def on_epoch_end(self, epoch, logs=None):
        self.generate("Shakespeare:", max_tokens=100, temperature=1.0)

In [29]:
# Tokenize starting prompt

text_generator = TextGenerator(vocab)

In [30]:
lstm.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=[text_generator],
)

Epoch 1/25
211/211 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - loss: 2.8845

NameError: name 'np' is not defined

In [ ]:
def print_probs(info, vocab, top_k=5):
    for i in info:
        print(f"\nPROMPT: {i['prompt']}")
        word_probs = i["word_probs"]
        p_sorted = np.sort(word_probs)[::-1][:top_k]
        i_sorted = np.argsort(word_probs)[::-1][:top_k]
        for p, i in zip(p_sorted, i_sorted):
            print(f"{vocab[i]}:   \t{np.round(100*p,2)}%")
        print("--------\n")

In [ ]:
info_1 = text_generator.generate(
    "Shall I compare thee to a summer's day?", max_tokens=10, temperature=1.0

)
info_2 = text_generator.generate(
    "To be, or not to be", max_tokens=10, temperature=1.0
)
info_3 = text_generator.generate(
    "All the world's a stage", max_tokens=10, temperature=1.0
)


In [ ]:
print_probs(info, vocab)

In [ ]:
info_1 = text_generator.generate(
    "Shall I compare thee to a summer's day?", max_tokens=10, temperature=0.2

)
info_2 = text_generator.generate(
    "To be, or not to be", max_tokens=10, temperature=0.2
)
info_3 = text_generator.generate(
    "All the world's a stage", max_tokens=10, temperature=0.2
)


In [ ]:
print_probs(info, vocab)

In [ ]:
info_1 = text_generator.generate(
    "Shall I compare thee to a summer's day?", max_tokens=8, temperature=0.2

)
info_2 = text_generator.generate(
    "To be, or not to be", max_tokens=8, temperature=0.2
)
info_3 = text_generator.generate(
    "All the world's a stage", max_tokens=8, temperature=0.2
)


In [ ]:
print_probs(info, vocab)

In [ ]:
info_1 = text_generator.generate(
    "Shall I compare thee to a summer's day?", max_tokens=8, temperature=1.0

)
info_2 = text_generator.generate(
    "To be, or not to be", max_tokens=8, temperature=1,0
)
info_3 = text_generator.generate(
    "All the world's a stage", max_tokens=8, temperature=1.0
)
